##Maximum Mariginal Relevance

---


MMR retrieves results that are relevant to the query while also being different from each other

###Step 1: Install Dependencies
These libraries help us interact with LLMs, generate embeddings, and store vectors.

In [ ]:
!pip install --quiet langchain-openai langchain-chroma


###Step 2: Imports & API Config
Allows our code to communicate with the OpenAI-compatible backend
* ChatOpenAI → to understand user query

* OpenAIEmbeddings → to convert text to vectors

* Chroma → vector database

* Pydantic → enforce structured output

In [ ]:
import os
from typing import List
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document



In [ ]:
os.environ["OPENAI_API_KEY"]="YOUR_API_KEY"
os.environ["OPENAI_API_BASE"]="https://apidev.navigatelabsai.com"

###Step 3: Documents
Content is used for semantic search, metadata can be used later for filtering

In [ ]:
docs=[
    Document(
        page_content="Scientists clone dinosaurs, chaos follows.",
        metadata={"year": 1993, "rating": 7.7, "genre": "sci-fi"}
    ),
    Document(
        page_content="A dream within a dream heist.",
        metadata={"year": 2010, "rating": 8.2, "genre": "sci-fi"}
    ),
    Document(
        page_content="Toys come alive when humans are away.",
        metadata={"year": 1995, "rating": 8.3, "genre": "animated"}
    ),
    Document(
        page_content="Detectives hunt a serial killer.",
        metadata={"year": 1995, "rating": 8.6, "genre": "crime"}
    ),
]


###Step 4: Embeddings + Vector Store
* Convert text into vectors
* Vector representations allow semantic similarity search.

In [ ]:
embeddings=OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://apidev.navigatelabsai.com"
)

vectorstore = Chroma.from_documents(docs, embeddings)

###Step 5: MMR Search
Retrieve documents using MMR

In [ ]:
def mmr_search(query, k=2, fetch_k=4, lambda_mult=0.5):
    """
    k: final number of documents returned
    fetch_k: initial pool size
    lambda_mult: balance between relevance and diversity
    """
    results=vectorstore.max_marginal_relevance_search(
        query=query,
        k=k,
        fetch_k=fetch_k,
        lambda_mult=lambda_mult
    )
    return results


###Step 6: Run MMR Retrieval
Execute MMR-based retrieval

In [ ]:
results = mmr_search(
    "sci-fi movie about dinosaurs",
    k=2, #Final results shown
    fetch_k=4, #Candidate pool
    lambda_mult=0.5 #Relevance vs diversity balance
)

print("\n--- MMR RESULTS ---")
for r in results:
    print(r.page_content)
    print("Metadata:", r.metadata)


###Summary:
**What Is Happening Internally**

* Retrieve top fetch_k similar documents

* Pick the most relevant one

* Pick next documents that are both relevant AND different

* Return final k results

* Retrieve top candidates → select most relevant → select next that adds new information → return diverse results